[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/06_your_own_classifier.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Your own classifier's scores

Allow about 25 minutes. What does the core analysis say about the classifier you
trained this week? You will join its scores to the CinC labels using pandas, run the
analysis on its development recordings, freeze two policies (a stated threshold and
the development optimum), and only then compare them on the evaluation recordings.

The action is **send this recording for AF review**, with the same prices, in
reviews, as the core notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and
$C_{TN}$ = 0. Attempt this on your own. Replace each `...` and then run the checker
beneath it. If you're stuck for about 10 minutes, use the stuck switch below the
problem (where there is one) or view the worked copy in `extension/solutions/`. Note
that a later cell that needs an unfinished problem will produce a `Waiting for ...`
message. Once that problem is complete or switched, click in the waiting cell,
choose **Runtime > Run before**, then run the waiting cell.

**Which scores to bring.** Please generate a CSV file with a header row
`recording,score` and a single row for each recording in the CinC dataset, with the
score produced by your model for AF. The notebook assigns your recordings to the
course's development and evaluation splits. If you do not provide a file, it will
use a demonstration column and notify you of this.

Please note that the comparison at the end will only be a 'true' held-out evaluation
if no evaluation recording influenced your model in any way. This includes model
training, pre-processing or feature selection, hyperparameter optimisation, early
stopping, model checkpointing or selection, recalibration, or an earlier choice of
threshold. For example, if you used the evaluation recordings as a validation set
for early stopping, this would not be a 'true' held-out evaluation. If you cannot
claim this, please leave `EVALUATION_UNTOUCHED = False` below and interpret the
result as a retrospective policy analysis. This will demonstrate how the frozen
policies behave on those recordings, not how they would perform on new ones.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from clinical_utility import calibration, data, own_scores, plots, reference
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)

# Upload your CSV in Colab's Files pane and give its name here, or paste its lines below.
SCORES_FILE = ""
PASTED_SCORES = """
"""
# True only if no evaluation recording influenced your model, as described above.
EVALUATION_UNTOUCHED = False

scores, note = own_scores.read_scores(SCORES_FILE, PASTED_SCORES)
print(note)
print(len(scores), "rows read.")
scores.head(3)

## Join the scores to the labels

`data.reference_labels()` is a DataFrame of all recording IDs from the CinC dataset
as well as the associated label: `A` (AF), `N`, `O` or `~` (noise). We need to join
this to the scores without making any assumptions. We will do this by checking each
row in the following order:

| Check | What to do with the row |
|---|---|
| Its ID appears more than once in your file | Exclude every copy; list the ID once in `duplicates`. |
| Its ID is not a CinC recording | Exclude it; list it in `unknown`. |
| Its score is not a number between zero and one | Exclude it; list it in `out_of_range`. |
| Its label is noise, `~` | Exclude it; count it in `noise`. |
| Otherwise | Keep it, with `af` true only for `A`. |

### Problem A: Make

Write `join_scores(scores, labels)` using pandas. This should output a dictionary
containing: `records` (a DataFrame containing `recording`, `score` and `af`, in the
same order as the input), `duplicates`, `unknown` and `out_of_range` (lists of IDs),
and `noise` (a count). You may find the following pandas functions useful:
`scores["recording"].duplicated(keep=False)`, `merge(labels, on="recording",
how="left", validate="one_to_one")` and `Series.between(0, 1)` (a NaN score will
evaluate to False).

In [ ]:
def join_scores(scores, labels):
    duplicated = scores["recording"].duplicated(keep=False)
    rest = scores[~duplicated].merge(labels, on="recording", how="left", validate="one_to_one")
    unknown = rest["reference_label"].isna()
    out_of_range = ~unknown & ~rest["score"].between(0, 1)
    noise = ~unknown & ~out_of_range & (rest["reference_label"] == "~")
    kept = rest[~(unknown | out_of_range | noise)]
    return {"records": pd.DataFrame({"recording": kept["recording"], "score": kept["score"],
                                     "af": kept["reference_label"] == "A"}).reset_index(drop=True),
            "duplicates": list(dict.fromkeys(scores.loc[duplicated, "recording"])),
            "unknown": rest.loc[unknown, "recording"].tolist(),
            "out_of_range": rest.loc[out_of_range, "recording"].tolist(),
            "noise": int(noise.sum())}
_ = check.run("own_join", join_scores)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# join_scores = check.use_reference("join_scores")

In [ ]:
# The join, then what each split can support. Evaluation outcomes stay hidden until the reveal.
empty = pd.DataFrame({"recording": [], "score": [], "af": []})
own = {"development": empty, "calibration": empty, "evaluation": empty}
if check.ready(join_scores=join_scores):
    joined = join_scores(scores, data.reference_labels())
    for key in ("duplicates", "unknown", "out_of_range"):
        print(f"{key}: {len(joined[key])}", joined[key][:5])
    print("noise recordings excluded:", joined["noise"])
    own = own_scores.with_splits(joined["records"])
    own_scores.describe_splits(own)

## The core analysis on development only

All analysis in this section is based only on the development recordings, and as
such any number calculated for a cutoff chosen here will be an *apparent* result:
the cutoff was derived from the same recordings, so they flatter it.

In [ ]:
best = None
development = own["development"]
if development.empty:
    check.waiting("development recordings: finish Problem A, or bring scores for recordings in "
                  "the development split")
else:
    points = reference.operating_points(development["af"], development["score"], costs)
    best = reference.cheapest(points)
    marks = {"Minimum loss": best}
    best_j = own_scores.max_j(points)
    if best_j is not None:
        marks["Maximum J"] = best_j
    n = len(development)
    display(pd.DataFrame({label: {"cutoff": row["threshold"], "reviews per 1,000": 1000 * row["reviews"] / n,
                                  "AF missed per 1,000": 1000 * row["FN"] / n, "loss": row["loss"]}
                          for label, row in marks.items()}).round(3))
    display(plots.objective_figure(points, marks))

In [ ]:
if best is None:
    check.waiting("the development analysis above, which needs development recordings")
else:
    compared = own_scores.stated_versus_optimum(development, costs, 0.2, best["threshold"])
    display(pd.DataFrame({"stated 0.2": compared["stated"], "minimum loss": compared["best"]}))
    print(f"Any cutoff {own_scores.interval_text(compared['interval'])} makes the minimum-loss "
          f"decisions; 0.2 is {'inside' if compared['same_decisions'] else 'outside'} that interval.")
    report = calibration.calibration_report(development["af"], development["score"]) \
        if development["af"].nunique() == 2 else None
    if report is not None:
        display(view.reliability({"Your scores": (report["observed"], report["predicted"])}))
        print("Brier:", round(report["brier"], 4))
    fit = own_scores.calibration_fit(development)
    for key, value in fit.items():
        if value is not None:
            print(f"Calibration {key}: {value:.3f}")

If your scores were perfectly calibrated probabilities, simply reviewing at the
threshold the prices imply, 0.2, would already be the right policy. A whole interval
of cutoffs results in the same decisions, and so even if the minimum-loss cutoff is
far from 0.2, the decisions will be the same if no score lies between them. You can
judge whether your model is calibrated by inspecting the reliability plot, as well
as the intercept and slope (when they can be estimated). Any difference between the
two policies on development will be apparent, as the optimum was chosen there.
Determining which policy is better is a question for the evaluation split.

## Freeze, then look

### Problem B: Make

Construct `frozen` by freezing two policies prior to the reveal. These policies are
`stated` (the threshold the prices imply) and `optimum` (the development
minimum-loss cutoff, `best["threshold"]`). `frozen` should be set to a dictionary
with the keys `stated`, `optimum`, `costs` and `p_t` (the threshold probability of
those prices).

In [ ]:
frozen = {"stated": reference.threshold_probability(costs),
          "optimum": float(best["threshold"]) if best is not None else None,
          "costs": dict(costs), "p_t": reference.threshold_probability(costs)}
frozen_ok = check.run("own_freeze", frozen, own["development"])

In [ ]:
# Reveal: both frozen policies on your evaluation recordings, and their paired difference.
if not frozen_ok:
    check.waiting("Problem B: freeze both policies on development")
elif own["evaluation"].empty:
    print("There are no evaluation recordings among your scores, so there is nothing to reveal.")
else:
    result = own_scores.compare_frozen(own["evaluation"], frozen)
    display(pd.concat({"stated threshold": result["stated"], "development optimum": result["optimum"]},
                      axis=1).round(3))
    difference = result["difference"]
    print(f"Net benefit, optimum minus stated: {difference['estimate']:.4f} "
          f"(95% interval {difference['low']:.4f} to {difference['high']:.4f}).")
    print(own_scores.evaluation_label(EVALUATION_UNTOUCHED))

Interpret the reveal as a comparison of the two frozen policies on recordings that
were not used to choose either policy. If the interval for the difference in net
benefit between the two policies contains zero, we can claim that these recordings
are not informative for choosing between the two policies. If there is a clear
advantage for the development optimum, it is a prompt to consider the calibration of
your scores (see Extension 1 for the repair). If data is limited on the number of AF
recordings, the bootstrap intervals may be untrustworthy, collapse (have zero width
if the two policies make the same decisions on the data), or be unavailable. In such
cases the number of AF recordings and the number of recordings that the policies
disagree on (printed with the reveal) should be reported instead of
over-interpreting a single result.

## Check your report

- Can you confirm if there was any data leakage between the evaluation set and your
  model? How can you be sure?
- What is the cutoff, number of reviews and number of AF missed for your model
  alongside the AUC value?
- Ensure that you clearly label the development results for a chosen cutoff as
  apparent performance.
- What is the difference in net benefit between the two frozen policies, and what is
  its interval?

**Question to consider:** what would you report about this classifier in addition to
the AUC, and what would a service need to know before acting on it?